# Category generation

Run **all cells** with the working directory set to the **repository root** (the folder that contains `pyproject.toml`).

This notebook:

- Defines property groups for **NPS Category**, **NPS Value Category**, **Has Numeric NPS**, and combined **NPS All**.
- Builds few-shot examples automatically when you pass `csv_path=...` and omit `examples` (see `ClassificationCategory.__init__` and `nps_crawling.config.Config`: `CLASSIFICATION_FEW_SHOT_NUM_EXAMPLES`, `CLASSIFICATION_FEW_SHOT_TEXT_COLUMN`, `CLASSIFICATION_FEW_SHOT_SAMPLE_SEED`, plus the same split/seed as evaluation: `CLASSIFICATION_RANDOM_SEED`, `CLASSIFICATION_GROUND_TRUTH_TEST_SIZE`).
- Instantiating each category writes JSON under `src/nps_crawling/classification/configurations/categories/<name>/<stable_id>.json` when that file does not exist yet.
- Pass `num_examples=` on a category to override the config default for that category only.
- The last cell syncs **NPS All** into `NPS_ALL_WITH_QWEN.json`.

**CLI:** From the repo root run python train/regenerate_category_configs.py to delete existing JSON in the NPS category folders (and TestCat), regenerate categories, and refresh NPS_ALL_WITH_QWEN.json.

**Tip:** To change how many few-shots all categories use by default, edit `Config.CLASSIFICATION_FEW_SHOT_NUM_EXAMPLES` in `config.py` (or set `num_examples=None` everywhere below to rely purely on config).

In [ ]:
# NPS Category Creation
from __future__ import annotations

import json
from pathlib import Path

from nps_crawling.classification.categories.category import (
    ClassificationCategory,
    ClassificationProperty,
    ClassificationType,
)

nps_category_properties = [
    ClassificationProperty(
        name="KPI_CURRENT_VALUE",
        description="The text reports a specific NPS value.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="KPI_TREND",
        description="The text describes change over time.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="KPI_HISTORICAL_COMPARISON",
        description="The text explicitly compares NPS against a previous period.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="TARGET_OUTLOOK",
        description="The text expresses a future goal, target, ambition, forecast, or expected future NPS.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="NPS_GOAL_REACHED",
        description="The text states that an NPS target or objective has been met or exceeded.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="METHODOLOGY_DEFINITION",
        description="The text explains what NPS is, how it is calculated, or what it measures.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="QUALITATIVE_ONLY",
        description="NPS is mentioned meaningfully but none of the other labels apply.",
        type=ClassificationType.BOOLEAN,
    ),
]

nps_value_category_properties = [
    ClassificationProperty(
        name="nps_value_fix",
        description="Direct NPS value.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_competition_industry",
        description="Industry benchmark or competitor NPS value.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_value_over",
        description="Threshold value associated with above, over, greater than, more than, or at least.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_value_below",
        description="Threshold value associated with below, under, less than, or at most.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_goal_value",
        description="Explicit NPS target value, only if value is at least 20.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_goal_change",
        description="Planned improvement amount such as improve by 10, increase by 5, or raise by 7.",
        type=ClassificationType.FLOAT,
    ),
]

has_numeric_nps_property = ClassificationProperty(
    name="has_numeric_nps",
    description="True if the text contains at least one explicit numeric NPS value.",
    type=ClassificationType.BOOLEAN,
)

all_classification_properties = [
    *nps_category_properties,
    *nps_value_category_properties,
    has_numeric_nps_property,
]

prompt_base = """You are an expert at analyzing text to identify and extract information about Net Promoter Score (NPS). NPS is a widely used metric that measures customer loyalty and satisfaction by asking customers how likely they are to recommend a product or service to others. Your task is to read the provided text and determine whether it contains specific information related to NPS, as defined by the following properties:"""

prompt_base_nps_category = (
    prompt_base
    + " Focus on the multi-label boolean dimensions only (whether each statement type applies)."
)
prompt_base_nps_value = (
    prompt_base
    + " Focus on extracting numeric NPS-related values only (direct value, benchmarks, thresholds, goals)."
)
prompt_base_has_numeric = (
    prompt_base
    + " Answer only whether the text states at least one concrete numeric NPS value."
)

In [ ]:
# Resolve repo root (works if cwd is repo root or train/).
from pathlib import Path


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "pyproject.toml").exists() and (p / "src" / "nps_crawling").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Run with cwd set to the NPS-Crawling repo root (contains pyproject.toml).")


PROJECT_ROOT = find_repo_root()
CSV_PATH = str((PROJECT_ROOT / "train" / "ground_truth_final.csv").resolve())

PIPELINE_JSON = (
    PROJECT_ROOT
    / "src"
    / "nps_crawling"
    / "classification"
    / "configurations"
    / "NPS_ALL_WITH_QWEN.json"
)

### Few-shot counts

Omit `num_examples` to use `Config.CLASSIFICATION_FEW_SHOT_NUM_EXAMPLES`. The combined **NPS All** task uses fewer examples below to keep prompts smaller.

In [ ]:
categories = {
    "NPS Category": ClassificationCategory(
        "NPS Category",
        nps_category_properties,
        prompt_base_nps_category,
        csv_path=CSV_PATH,
    ),
    "NPS Value Category": ClassificationCategory(
        "NPS Value Category",
        nps_value_category_properties,
        prompt_base_nps_value,
        csv_path=CSV_PATH,
    ),
    "Has Numeric NPS": ClassificationCategory(
        "Has Numeric NPS",
        [has_numeric_nps_property],
        prompt_base_has_numeric,
        csv_path=CSV_PATH,
        num_examples=6,
    ),
    "NPS All": ClassificationCategory(
        "NPS All",
        all_classification_properties,
        prompt_base,
        csv_path=CSV_PATH,
        num_examples=5,
    ),
}

for label, cat in categories.items():
    print(
        f"{label}:\n  {cat.config_path}\n"
        f"  num_examples_field={cat.num_examples!r}  len(examples)={len(cat.examples)}  stable_id={cat.stable_id}\n"
    )

In [ ]:
# Sync embedded "NPS All" into the Qwen pipeline JSON.
if PIPELINE_JSON.exists():
    with open(PIPELINE_JSON, encoding="utf-8") as f:
        pipeline = json.load(f)
    for item in pipeline.get("classification_configuration", []):
        cat = item.get("category")
        if isinstance(cat, dict) and cat.get("name") == "NPS All":
            item["category"] = categories["NPS All"].to_dict()
            break
    with open(PIPELINE_JSON, "w", encoding="utf-8") as f:
        json.dump(pipeline, f, indent=2, ensure_ascii=False)
    print("Updated pipeline:", PIPELINE_JSON)
else:
    print("Pipeline file not found, skipped:", PIPELINE_JSON)